# A/B Testing Sample Size — $E[X]$, $\text{Var}(X)$, and the Role of Variance

You are testing two versions of a recommendation algorithm:

- **Control (A)**: current algorithm, click-through rate (CTR) = **5%**.
- **Treatment (B)**: new algorithm, hoped CTR = **7%**.

Each user either clicks (1) or does not (0). So CTR is the mean of a **Bernoulli random variable**:

$$
X \sim \text{Bernoulli}(p) \qquad E[X] = p \qquad \text{Var}(X) = p(1-p)
$$

Questions:

1. What are $E[X]$ and $\text{Var}(X)$ for each variant?
2. Why does $\text{Var}(X)$ directly determine the sample size needed?
3. How many users are needed to reliably detect this difference?
4. What happens if the true effect is smaller than expected?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

p_a = 0.05  # control CTR
p_b = 0.07  # treatment CTR

e_a, var_a = p_a, p_a * (1 - p_a)
e_b, var_b = p_b, p_b * (1 - p_b)

print(f"Control   (A): E[X] = {e_a:.3f}  Var(X) = {var_a:.4f}  Std = {np.sqrt(var_a):.4f}")
print(f"Treatment (B): E[X] = {e_b:.3f}  Var(X) = {var_b:.4f}  Std = {np.sqrt(var_b):.4f}")
print(f"\nTrue difference in CTR: {e_b - e_a:.3f} ({(e_b - e_a)/e_a:.0%} relative lift)")

## Step 1: Plot the PMFs of the two variants

Each user's click is Bernoulli — a PMF over $\{0, 1\}$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (label, p) in zip(axes, [("A (control, p=5%)", p_a), ("B (treatment, p=7%)", p_b)]):
    ax.bar([0, 1], [1 - p, p], width=0.4)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No click (0)", "Click (1)"])
    ax.set_ylabel("Probability")
    ax.set_title(f"Bernoulli PMF — Variant {label}")
    ax.set_ylim(0, 1)
    ax.axhline(p, linestyle="--", alpha=0.5, label=f"E[X] = {p:.2f}")
    ax.legend()

plt.tight_layout()
plt.show()

## Step 2: Why variance determines sample size

When we estimate the CTR from $n$ users, the **sample mean** $\bar{X}$ has:

$$
E[\bar{X}] = p \qquad \text{Var}(\bar{X}) = \frac{\text{Var}(X)}{n} = \frac{p(1-p)}{n}
$$

The standard error shrinks as $1/\sqrt{n}$. To detect a true difference $\delta = p_B - p_A$, we need the sampling noise to be much smaller than $\delta$.

The standard two-proportion sample size formula:

$$
n = \frac{(z_{\alpha/2} + z_\beta)^2 \cdot (\text{Var}(X_A) + \text{Var}(X_B))}{\delta^2}
$$

where $z_{\alpha/2} = 1.96$ (95% confidence) and $z_\beta = 0.84$ (80% power).

In [ ]:
z_alpha = 1.96  # 95% confidence (two-sided)
z_beta = 0.84   # 80% power
delta = p_b - p_a

n_required = ((z_alpha + z_beta)**2 * (var_a + var_b)) / delta**2
n_required = int(np.ceil(n_required))

print(f"Effect size (delta):  {delta:.3f}")
print(f"Var(X_A) + Var(X_B):  {var_a + var_b:.4f}")
print(f"\nSample size needed per variant: {n_required:,} users")
print(f"Total users needed:             {2 * n_required:,} users")

## Step 3: Simulate — does the test reliably detect the effect?

Run the experiment many times and measure how often we correctly detect that B is better.

In [ ]:
def run_experiment(n, p_a, p_b, n_trials=5000):
    clicks_a = rng.binomial(n, p_a, size=n_trials)
    clicks_b = rng.binomial(n, p_b, size=n_trials)
    ctr_a = clicks_a / n
    ctr_b = clicks_b / n
    # Fraction of trials where B appears better
    detected = np.mean(ctr_b > ctr_a)
    return detected

# Test at the required sample size and at smaller sizes
sample_sizes = [100, 500, 1000, n_required // 2, n_required, n_required * 2]

print(f"True CTR A = {p_a:.1%}, True CTR B = {p_b:.1%}")
print(f"{'Sample size per variant':>28} | {'% times B detected as better':>28}")
print("-" * 60)
for n in sample_sizes:
    power = run_experiment(n, p_a, p_b)
    marker = " ← required" if n == n_required else ""
    print(f"{n:>28,} | {power:>27.1%}{marker}")

## Step 4: What if the true effect is smaller?

The smaller the true difference $\delta$, the larger the sample needed — because $\delta^2$ is in the denominator.

In [ ]:
deltas = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10]
n_sizes = []

for d in deltas:
    p_b_hyp = p_a + d
    var_b_hyp = p_b_hyp * (1 - p_b_hyp)
    n = ((z_alpha + z_beta)**2 * (var_a + var_b_hyp)) / d**2
    n_sizes.append(int(np.ceil(n)))

plt.plot(deltas, n_sizes, marker="o")
plt.xlabel("True effect size δ = p_B − p_A")
plt.ylabel("Sample size needed per variant")
plt.title("Sample size grows as 1/δ² — small effects require massive tests")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

for d, n in zip(deltas, n_sizes):
    print(f"δ = {d:.3f}  →  n = {n:>8,} per variant  ({2*n:>9,} total)")

## Observation

- Each user's click is a **Bernoulli PMF**: $E[X] = p$ is the CTR, $\text{Var}(X) = p(1-p)$ is the noise per user.
- The **sample mean** $\bar{X}$ has variance $\text{Var}(X)/n$ — collecting more data reduces uncertainty, but only as $1/\sqrt{n}$.
- **Sample size is driven by variance**: the formula $n \propto \text{Var}(X) / \delta^2$ shows that high variance or small effects require exponentially more data.
- Detecting a 2% absolute CTR lift (5% → 7%) requires thousands of users per variant. Detecting a 0.5% lift requires hundreds of thousands.
- This is why large tech companies run A/B tests on millions of users — not because their models are uncertain, but because $\text{Var}(X)$ is fixed by the Bernoulli structure and small improvements have small $\delta$.